# AgriSmart AI — Model 2 Field Adaptation Master Training Notebook (v2)
=======================================================================

**Thin Orchestration Notebook for Google Colab T4 GPU Runtimes**

This notebook is a **thin orchestrator** wrapper. All business logic, dataset download/preparation,
leakage checks, guardrails, and model training are encapsulated in the master CLI runner:
`scripts/run_model2_colab.py`.

### Quick Start:
1. Connect to a **T4 GPU Runtime** (Runtime -> Change runtime type -> T4 GPU).
2. Click **Runtime -> Run all**.

---

### CELL 1 — ENVIRONMENT SETUP & DEPENDENCY INSTALLATION
Installs required Python packages quietly and configures environment.

In [ ]:
# Cell 1: Install Dependencies
import sys
import subprocess

# Safe UTF-8 output
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

print("[CELL 1] Installing required packages...")
reqs = ["timm", "huggingface_hub", "albumentations", "scikit-learn", "tqdm", "pillow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + reqs, check=True)
print("[OK] Dependencies installed successfully.")

### CELL 2 — CLONE REPOSITORY & WORKSPACE SETUP
Clones the AgriSmart-AI repository into `/content/AgriSmart-AI` or verifies local repository root.

In [ ]:
# Cell 2: Workspace Setup & Repository Cloning
import os
from pathlib import Path

repo_url = "https://github.com/Parrthiv125/AgriSmart-AI.git"
colab_target = Path("/content/AgriSmart-AI")

if colab_target.exists():
    REPO_ROOT = colab_target
    os.chdir(REPO_ROOT)
    subprocess.run(["git", "checkout", "main"], check=True)
    subprocess.run(["git", "pull", "origin", "main"], check=True)
else:
    if Path("models/classes.json").exists():
        REPO_ROOT = Path(os.getcwd()).resolve()
    else:
        print(f"Cloning {repo_url} into {colab_target}...")
        subprocess.run(["git", "clone", repo_url, str(colab_target)], check=True)
        REPO_ROOT = colab_target
        os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"[OK] Workspace Root: {REPO_ROOT}")
print(f"[OK] Commit Hash:    {commit_hash}")

### CELL 3 — CUDA GPU DETECTION & VALIDATION
Verifies CUDA GPU availability for training.

In [ ]:
# Cell 3: GPU Detection & Validation
import torch

print("[CELL 3] Detecting GPU device...")
if not torch.cuda.is_available():
    raise RuntimeError("[CRITICAL ERROR] CUDA GPU not detected! Change runtime type: Runtime -> Change runtime type -> T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
print(f"[OK] CUDA GPU Detected: {gpu_name}")
print(f"     PyTorch Version:  {torch.__version__}")

### CELL 4 — EXECUTE MASTER MODEL 2 ORCHESTRATOR
Runs the single authoritative master runner (`scripts/run_model2_colab.py`).

In [ ]:
# Cell 4: Execute Master Model 2 Runner
import sys
import subprocess

print("[CELL 4] Launching master Model 2 runner (scripts/run_model2_colab.py)...")
print("======================================================================")

cmd = [sys.executable, "scripts/run_model2_colab.py"]
subprocess.run(cmd, check=True)

### CELL 5 — FINAL ARTIFACT SUMMARY & GUARDRAIL AUDIT
Verifies training artifacts, Model 1 SHA256 integrity, and locked test set isolation.

In [ ]:
# Cell 5: Final Artifact Summary & Verification
import json
from pathlib import Path
from data.common import (
    MODEL1_CKPT_PATH,
    EXPECTED_MODEL1_HASH,
    MODEL2_CKPT_OUT,
    MODEL2_LAST_OUT,
    MODEL2_METADATA_OUT,
    PD_TEST_DIR,
    PV_TEST_DIR,
    compute_file_sha256,
)

print("=" * 70)
print(" AGRISMART AI — FINAL MODEL 2 ARTIFACT SUMMARY")
print("=" * 70)

if MODEL2_METADATA_OUT.exists():
    with open(MODEL2_METADATA_OUT, "r", encoding="utf-8") as f:
        meta = json.load(f)
    t_res = meta.get("training_results", {})
    s_strat = meta.get("sampling_strategy", {})
    print(f"  Architecture:                {meta.get('architecture', 'efficientnet_b2')}")
    print(f"  Best Epoch:                  {t_res.get('best_epoch')}")
    print(f"  Best Validation Macro-F1:    {t_res.get('best_val_macro_f1', 0.0):.4f}")
    print(f"  Sampling Method:             {s_strat.get('method')}")
    print(f"  PlantDoc Oversample Factor:  {s_strat.get('plantdoc_oversample_factor')}x")

m1_hash = compute_file_sha256(MODEL1_CKPT_PATH)
print(f"  Model 1 SHA256 Integrity:    {'VERIFIED UNTOUCHED' if m1_hash == EXPECTED_MODEL1_HASH else 'FAILED Altered!'}")
print(f"  Best Checkpoint Saved:       {MODEL2_CKPT_OUT.relative_to(REPO_ROOT)} ({MODEL2_CKPT_OUT.stat().st_size / 1e6:.1f} MB)")
if MODEL2_LAST_OUT.exists():
    print(f"  Last Checkpoint Saved:       {MODEL2_LAST_OUT.relative_to(REPO_ROOT)} ({MODEL2_LAST_OUT.stat().st_size / 1e6:.1f} MB)")

pd_test_count = len(list(PD_TEST_DIR.rglob("*"))) if PD_TEST_DIR.exists() else 0
pv_test_count = len(list(PV_TEST_DIR.rglob("*"))) if PV_TEST_DIR.exists() else 0
print(f"  PlantDoc TEST Status:        LOCKED & UNTOUCHED ({pd_test_count} items)")
print(f"  PlantVillage TEST Status:     LOCKED & UNTOUCHED ({pv_test_count} items)")
print(f"  PlantDoc TEST Evaluated:     FALSE (Evaluation locked until model frozen)")
print("=" * 70)
print(" [SUCCESS] ALL MODEL 2 ARTIFACTS VERIFIED SUCCESSFULLY.")